# 🧬 Preprocessing Notebook 3 — Genomic Data
## Ancestry-Calibrated Polygenic Score Harmonization Pipeline
### PGS Catalog × GenomeIndia | Coordinate-Based Matching + Allele Alignment

---

> **Project Goal:** Build a population-aware genetic risk map for cardiovascular disease (CVD)  
> by harmonizing a published Polygenic Score (PGS) — derived from predominantly European GWAS  
> cohorts — with allele frequency data from the **GenomeIndia (GI)** reference panel, which  
> captures the genetic diversity of 9,768 individuals across the Indian subcontinent.

> **Why this matters:** A polygenic score is only as accurate as its calibration to the  
> target population. Effect sizes and risk allele frequencies from European GWAS cohorts  
> can be systematically biased when applied to South Asians, due to differences in LD  
> structure, ancestral haplotype blocks, and population-specific allele frequency drift.  
> This notebook corrects for that bias by re-anchoring each PGS SNP to its **observed  
> frequency in Indian ancestry**.

---

## 📌 What this pipeline does (in order)
1. Mounts Google Drive and validates all file paths
2. Loads and validates the PGS Catalog scoring file (PGS002809)
3. Harmonizes PGS SNPs against GenomeIndia reference panel (coordinate-based matching + allele alignment)
4. **[FIX]** Logs exactly which PGS SNPs were dropped and why (required for paper methods section)
5. Applies two bioinformatic QC filters (Palindromic SNP filter + MAF filter)
6. Exports the final ancestry-calibrated genetic map

---

## 📂 Required inputs

```
CVD_DigitalTwin_Project/
└── data/
    └── raw/
        ├── pgs_catalog_2809.tsv                      ← PGS Catalog score file
        └── genome_india/
            ├── GI_9768_CBR-NIBMG_JointCall_AF_chr1.tsv
            ├── GI_9768_CBR-NIBMG_JointCall_AF_chr2.tsv
            └── ... (22 files, chr1 through chr22)
```

| Parameter | Value |
|---|---|
| PGS Catalog Score | PGS002809 (Cardiovascular Disease) |
| Reference Panel | GenomeIndia 9,768-sample WGS |
| Total GI Variants | ~129.9 million |
| Harmonization Strategy | Coordinate-Based Positional Matching + Allele Alignment |
| QC Filters | Palindromic SNP Filter + Minor Allele Frequency (MAF) Filter |


> **Edit log — 23/7/26 (July 23, 2026):** NB3 checklist items fixed:
> - Phase 5 markdown no longer hardcodes "184 PGS SNPs" — it referenced a stale run's count. Reworded to point at `INITIAL_PGS_SNP_COUNT` (Phase 2's dynamically printed value, currently 205 on this run's data) instead of a number typed into markdown, so it can't go stale again if the input PGS file changes size.
> - Phase 3: added a hard assertion right after the position-based merge (`merged['rsID'].is_unique`) to catch duplicate/fan-out matches — e.g. a multi-allelic GI site at the same position as a PGS SNP would otherwise silently inflate `total_position_matches` and duplicate that SNP's row in the final output. Verified against a synthetic test: passes on clean data, raises a descriptive error on a deliberately constructed fan-out case.
>  *- Rerun/SNP-count confirmation (~182 retained) is **not verified by this edit** — this notebook needs the real 22-chromosome GenomeIndia files and the real PGS catalog file, both only available in your Drive. Please rerun in Colab and share the Phase 4/5 output for a final check.*


---
# PHASE 1: Environment Setup

## Architectural principle
All paths declared once as `ALL_CAPS` constants. No hardcoded paths appear anywhere else  
in this notebook. Changing `PROJECT_ROOT` is sufficient to redirect the entire pipeline.

> 📝 **Edit `PROJECT_ROOT`** to match your Google Drive structure before running.


In [1]:
import os
import glob
import pickle
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np

# ── Mount Drive ─────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ── SINGLE SOURCE OF TRUTH ──────────────────────────────────
BASE_DIR = "/content/drive/MyDrive/CAD_DT_Final/"

RAW_DIR        = BASE_DIR + "Data/Raw/"
GI_DIR         = RAW_DIR + "Genome_India/"
OUTPUTS_DIR    = BASE_DIR + "Outputs/"
GENETICS_DIR   = OUTPUTS_DIR + "Genetics/"

# ── File Paths ──────────────────────────────────────────────
PGS_PATH         = RAW_DIR + "pgs_catalog_2809.tsv"
OUTPUT_PATH      = GENETICS_DIR + "harmonized_genetic_map.csv"
DROPPED_SNPS_LOG = GENETICS_DIR + "dropped_snps_audit_log.csv"

# ── Create required folders ─────────────────────────────────
os.makedirs(GENETICS_DIR, exist_ok=True)

# ── Pre-flight validation ───────────────────────────────────
print("=" * 65)
print("  CVD DIGITAL TWIN — PGS HARMONIZATION PIPELINE")
print("  PHASE 1: Environment Initialization")
print("=" * 65)

print(f"  BASE_DIR        : {BASE_DIR}")
print(f"  PGS_PATH        : {PGS_PATH}")
print(f"  GI_DIR          : {GI_DIR}")
print(f"  OUTPUT_DIR      : {GENETICS_DIR}")

assert os.path.isfile(PGS_PATH), f"❌ PGS file not found: {PGS_PATH}"
assert os.path.isdir(GI_DIR), f"❌ GenomeIndia folder not found: {GI_DIR}"

gi_file_list = sorted(glob.glob(GI_DIR + "*.tsv"))

print(f"  GenomeIndia files found: {len(gi_file_list)} / 22")

if len(gi_file_list) < 22:
    print(f"  ⚠️ WARNING: Expected 22 chromosome files, found {len(gi_file_list)}")

print("\n[PHASE 1 COMPLETE] ✅ Environment ready.")
print("=" * 65)

Mounted at /content/drive
  CVD DIGITAL TWIN — PGS HARMONIZATION PIPELINE
  PHASE 1: Environment Initialization
  BASE_DIR        : /content/drive/MyDrive/CAD_DT_Final/
  PGS_PATH        : /content/drive/MyDrive/CAD_DT_Final/Data/Raw/pgs_catalog_2809.tsv
  GI_DIR          : /content/drive/MyDrive/CAD_DT_Final/Data/Raw/Genome_India/
  OUTPUT_DIR      : /content/drive/MyDrive/CAD_DT_Final/Outputs/Genetics/
  GenomeIndia files found: 22 / 22

[PHASE 1 COMPLETE] ✅ Environment ready.


---
# PHASE 2: PGS Data Loading & Validation

## What is a PGS Catalog Score?

A **Polygenic Score (PGS)** aggregates the small, additive effects of genetic variants (SNPs)  
into a single numeric risk estimate. The PGS Catalog (https://www.pgscatalog.org) is the  
gold-standard repository for published, peer-reviewed polygenic scores.

**PGS002809** is a CVD risk score. Each row represents one SNP with:

| Column | Description |
|---|---|
| `rsID` | dbSNP reference identifier (e.g., rs11591147) |
| `chr_name` | Chromosome (1–22, X) |
| `chr_position` | GRCh37/38 base-pair coordinate |
| `effect_allele` | The allele that *increases* CVD risk |
| `effect_weight` | Log-odds coefficient (β) from the GWAS model |

The `effect_weight` encodes direction and magnitude. A weight of 0.547 for rs186696265  
represents a very large per-allele log-odds increase in CVD risk — this SNP alone substantially  
shifts the probabilistic risk estimate for any carrier.


In [2]:
print("=" * 65)
print("  PHASE 2: PGS Data Loading & Validation")
print("=" * 65)

PGS_COLUMNS_REQUIRED = ['rsID', 'chr_name', 'chr_position', 'effect_allele', 'effect_weight']

pgs_raw = pd.read_csv(
    PGS_PATH,
    sep='\t',
    comment='#',
    usecols=PGS_COLUMNS_REQUIRED
)

print(f"  ✅ PGS file loaded: {pgs_raw.shape[0]} SNPs × {pgs_raw.shape[1]} columns")

# Standardise chr_name to string without 'chr' prefix
pgs_raw['chr_name'] = pgs_raw['chr_name'].astype(str).str.replace(r'^chr', '', regex=True)

# Uppercase alleles for case-insensitive matching
pgs_raw['effect_allele'] = pgs_raw['effect_allele'].str.upper()

# Drop rows missing core fields
n_before = len(pgs_raw)
pgs_clean = pgs_raw.dropna(subset=['chr_name', 'chr_position', 'effect_allele', 'effect_weight']).copy()
n_dropped_missing = n_before - len(pgs_clean)

if n_dropped_missing > 0:
    print(f"  ⚠️  Dropped {n_dropped_missing} SNPs with missing core fields.")
else:
    print(f"  ✅ No SNPs dropped for missing values ({len(pgs_clean)} complete).")

INITIAL_PGS_SNP_COUNT = len(pgs_clean)

print()
print("  Effect weight (β) distribution:")
w = pgs_clean['effect_weight']
print(f"    Min={w.min():.4f}  Mean={w.mean():.4f}  Median={w.median():.4f}  Max={w.max():.4f}")
top_snp = pgs_clean.loc[w.idxmax()]
print(f"  🔬 Largest effect SNP: {top_snp['rsID']}  β={top_snp['effect_weight']:.4f}")

print()
print(f"[PHASE 2 COMPLETE] ✅ {INITIAL_PGS_SNP_COUNT} PGS SNPs ready for harmonization.")
print("=" * 65)


  PHASE 2: PGS Data Loading & Validation
  ✅ PGS file loaded: 205 SNPs × 5 columns
  ✅ No SNPs dropped for missing values (205 complete).

  Effect weight (β) distribution:
    Min=0.0220  Mean=0.0643  Median=0.0518  Max=0.5467
  🔬 Largest effect SNP: rs186696265  β=0.5467

[PHASE 2 COMPLETE] ✅ 205 PGS SNPs ready for harmonization.


---
# PHASE 3: Multi-File Harmonization Engine

## Why coordinate-based matching instead of rsID matching?

Matching by rsID is intuitive but unreliable across databases:
1. The same position can have different rsIDs in different database versions (rsID merging/splitting events in dbSNP).
2. Population-specific databases like GenomeIndia contain novel variants without rsIDs.
3. Build mismatch (GRCh37 vs GRCh38) can cause rsID→position mismatches.

**Solution:** Match on **(Chromosome + Genomic Position)** — a composite key that is  
build-stable and database-agnostic. This is the gold-standard approach used by the  
PGS Catalog harmonization pipeline and tools like PLINK2 and LiftOver.

## Why directional allele alignment matters

Knowing *which* position to look up is only half the problem. We also need to establish  
whether the PGS **effect allele** is the ALT or REF allele in the GenomeIndia VCF:

- **Effect allele = ALT:** `risk_freq = Alt_AF` (direct)
- **Effect allele = REF:** `risk_freq = 1 − Alt_AF` (flip required)
- **Neither matches:** SNP is multi-allelic or has a build mismatch → **dropped**

Without this directional alignment, a SNP with risk allele frequency 0.85 could be  
incorrectly encoded as 0.15 — a near-complete inversion. Across hundreds of SNPs,  
such errors compound to produce a clinically meaningless polygenic score.

### Diagnostic cell — run this first to confirm GI file format


In [3]:
# ── DIAGNOSTIC + HEADER VALIDATION — sample 3 GI files ──────────────────────
gi_file_list = sorted(glob.glob(GI_DIR + "*.tsv"))
if len(gi_file_list) == 0:
    raise FileNotFoundError(f"No .tsv files in GI_DIR: {GI_DIR}. Check your path.")

# Sample up to 3 files to confirm consistent column count across chromosomes
sample_files = gi_file_list[:min(3, len(gi_file_list))]
col_counts = {}

for fpath in sample_files:
    fname = os.path.basename(fpath)
    sample = pd.read_csv(fpath, sep='\t', nrows=5, header=None)
    col_counts[fname] = sample.shape[1]
    print(f"File: {fname}  →  {sample.shape[1]} columns")
    for i, col in enumerate(sample.columns):
        print(f"  [{i}] sample values: {sample[col].tolist()}")
    print()

# Assert all sampled files have the same number of columns
unique_col_counts = set(col_counts.values())
assert len(unique_col_counts) == 1, (
    f"❌ Inconsistent column counts across GI files: {col_counts}\n"
    "Check for header rows or format differences between chromosomes."
)
print(f"✅ All sampled files have {list(unique_col_counts)[0]} columns — consistent format confirmed.")
print("   header=None is safe to use in Phase 3.")
print()
print("Expected column layout (if no header):")
print("  [0] chromosome  [1] position  [2] variant_id  [3] ref_allele  [4] alt_allele  [5] alt_af")
print()
print("If the above does not match, update GI_COLUMN_NAMES in Phase 3 before running.")


File: GI_9768_CBR-NIBMG_JointCall_AF_chr1.tsv  →  6 columns
  [0] sample values: ['chr1', 'chr1', 'chr1', 'chr1', 'chr1']
  [1] sample values: [15835, 15849, 16288, 16378, 17746]
  [2] sample values: ['.', '.', '.', '.', '.']
  [3] sample values: ['G', 'C', 'C', 'T', 'A']
  [4] sample values: ['A', 'T', 'G', 'C', 'G']
  [5] sample values: [0.000258291, 0.00310366, 0.0230604, 0.123834, 0.000871616]

File: GI_9768_CBR-NIBMG_JointCall_AF_chr10.tsv  →  6 columns
  [0] sample values: ['chr10', 'chr10', 'chr10', 'chr10', 'chr10']
  [1] sample values: [10651, 11602, 13398, 16143, 18828]
  [2] sample values: ['.', '.', '.', '.', '.']
  [3] sample values: ['A', 'C', 'G', 'G', 'T']
  [4] sample values: ['C', 'A', 'C', 'T', 'C']
  [5] sample values: [0.998749, 0.033525, 0.00429799, 0.0969262, 0.15087]

File: GI_9768_CBR-NIBMG_JointCall_AF_chr11.tsv  →  6 columns
  [0] sample values: ['chr11', 'chr11', 'chr11', 'chr11', 'chr11']
  [1] sample values: [102813, 128802, 128964, 129012, 129013]
  [2] s

In [ ]:
# ── CELL 3.1 — Multi-File Harmonization Engine ───────────────────────────────

print("=" * 65)
print("  PHASE 3: Multi-File Harmonization Engine")
print("  Strategy: Coordinate-Based Matching + Directional Allele Alignment")
print("=" * 65)

# Column names for GenomeIndia files — confirmed via diagnostic cell above
# GI files have NO header row; columns are assigned by position
GI_COLUMN_NAMES = [
    'gi_chromosome',
    'gi_position',
    'gi_variant_id',
    'gi_ref_allele',
    'gi_alt_allele',
    'gi_alt_allele_frequency'
]

pgs_clean['chr_position'] = pgs_clean['chr_position'].astype(int)

harmonized_results = []   # per-chromosome DataFrames

# FIX: Comprehensive per-SNP drop audit
# We track every PGS SNP and the reason it was excluded (if any)
# This is required for the paper's methods section and for reproducibility
pgs_clean['drop_reason'] = 'retained'  # default; updated below as SNPs are processed

total_chromosomes_processed      = 0
total_position_matches           = 0
total_allele_matched_alt         = 0
total_allele_matched_ref         = 0
total_dropped_no_allele_match    = 0
dropped_no_position_match_rsids  = []
dropped_no_allele_match_rsids    = []

print(f"\n  Processing {len(gi_file_list)} chromosome files...")
print("-" * 65)

for gi_filepath in gi_file_list:
    gi_filename  = os.path.basename(gi_filepath)
    chr_label    = gi_filename.split('_chr')[-1].replace('.tsv', '')

    print(f"  🔬 Chromosome {chr_label:>2} | File: {gi_filename}")

    try:
        gi_df = pd.read_csv(
            gi_filepath,
            sep='\t',
            header=None,
            names=GI_COLUMN_NAMES,
            dtype={'gi_position': int, 'gi_alt_allele_frequency': float}
        )
    except Exception as e:
        print(f"     ⚠️  Could not read file. Skipping. Error: {e}")
        continue

    gi_df['gi_ref_allele'] = gi_df['gi_ref_allele'].str.upper()
    gi_df['gi_alt_allele'] = gi_df['gi_alt_allele'].str.upper()

    pgs_chr = pgs_clean[pgs_clean['chr_name'] == chr_label].copy()
    print(f"     GI variants: {len(gi_df):,}  |  PGS SNPs targeting this chr: {len(pgs_chr)}")

    if len(pgs_chr) == 0:
        print(f"     ⏭️  No PGS SNPs on chr{chr_label}. Skipping.")
        continue

    # Track SNPs on this chromosome that got no positional match
    merged = pd.merge(
        pgs_chr,
        gi_df[['gi_position', 'gi_ref_allele', 'gi_alt_allele', 'gi_alt_allele_frequency']],
        left_on='chr_position',
        right_on='gi_position',
        how='inner'
    )

    # FIX (added 2026-07-23): guard against duplicate/fan-out matches.
    # If GI has >1 row at the same position (a multi-allelic site split across
    # rows), a single PGS SNP can match more than once here, silently
    # inflating total_position_matches and duplicating that SNP's row in the
    # final harmonized map. Fail loudly instead of letting it pass silently.
    if not merged['rsID'].is_unique:
        dup_counts = merged['rsID'].value_counts()
        dup_rsids = dup_counts[dup_counts > 1]
        raise AssertionError(
            f"[FATAL] Duplicate rsID matches on chr{chr_label} after the position-based "
            f"merge: {len(merged)} rows for {merged['rsID'].nunique()} unique rsIDs. "
            f"Likely a multi-allelic site in the GI file at a position also hit by a PGS SNP. "
            f"Affected rsIDs: {dup_rsids.to_dict()}"
        )

    # Record which rsIDs got NO positional match
    matched_rsids = set(merged['rsID'])
    no_pos_match  = pgs_chr[~pgs_chr['rsID'].isin(matched_rsids)]['rsID'].tolist()
    dropped_no_position_match_rsids.extend(no_pos_match)
    for rsid in no_pos_match:
        pgs_clean.loc[pgs_clean['rsID'] == rsid, 'drop_reason'] = 'no_positional_match_in_gi'

    total_position_matches += len(merged)
    print(f"     Positional matches: {len(merged)}")

    if len(merged) == 0:
        print(f"     ⚠️  Zero matches. Check genome build compatibility (GRCh37 vs GRCh38).")
        continue

    # Directional allele alignment
    cond_alt = merged['effect_allele'] == merged['gi_alt_allele']
    cond_ref = merged['effect_allele'] == merged['gi_ref_allele']
    cond_none = ~(cond_alt | cond_ref)

    merged['adjusted_risk_freq'] = np.select(
        [cond_alt, cond_ref],
        [merged['gi_alt_allele_frequency'], 1 - merged['gi_alt_allele_frequency']],
        default=np.nan
    )

    n_alt  = int(cond_alt.sum())
    n_ref  = int(cond_ref.sum())
    n_none = int(cond_none.sum())
    total_allele_matched_alt          += n_alt
    total_allele_matched_ref          += n_ref
    total_dropped_no_allele_match     += n_none

    no_allele_rsids = merged.loc[cond_none, 'rsID'].tolist()
    dropped_no_allele_match_rsids.extend(no_allele_rsids)
    for rsid in no_allele_rsids:
        pgs_clean.loc[pgs_clean['rsID'] == rsid, 'drop_reason'] = 'no_allele_match_multiallellic_or_indel'

    print(f"     Allele align — Effect=ALT: {n_alt} | Effect=REF(flip): {n_ref} | No match (dropped): {n_none}")

    aligned = merged.dropna(subset=['adjusted_risk_freq']).copy()
    harmonized_results.append(aligned)
    total_chromosomes_processed += 1

print()
print("-" * 65)
print("[STEP 3.2] Harmonization Engine Summary:")
print(f"  Chromosomes processed      : {total_chromosomes_processed}")
print(f"  Total positional matches   : {total_position_matches:,}")
print(f"  Effect=ALT alignments      : {total_allele_matched_alt:,}")
print(f"  Effect=REF (flip) aligns   : {total_allele_matched_ref:,}")
print(f"  Dropped (no allele match)  : {total_dropped_no_allele_match:,}")
print(f"  Dropped (no pos match)     : {len(dropped_no_position_match_rsids):,}")
print()

# ── GENOME BUILD SAFETY CHECK ────────────────────────────────────────────────
# If match_rate < 0.80, the PGS and GI files are likely on different genome builds
# (e.g. GRCh37 vs GRCh38). This would produce silently wrong results downstream.
match_rate = total_position_matches / INITIAL_PGS_SNP_COUNT if INITIAL_PGS_SNP_COUNT > 0 else 0
print(f"  Genome build match rate    : {match_rate*100:.1f}%  ({total_position_matches} / {INITIAL_PGS_SNP_COUNT})")
assert match_rate > 0.80, (
    f"[FATAL] Genome build mismatch suspected: only {match_rate*100:.1f}% of PGS SNPs "
    "matched positionally in GenomeIndia. Verify both files use the same build (GRCh37 or GRCh38)."
)
print(f"  ✅ Genome build check passed (match_rate={match_rate*100:.1f}% > 80%)")
print()
print("  SNPs with NO positional match in GenomeIndia:")
if dropped_no_position_match_rsids:
    for rsid in dropped_no_position_match_rsids:
        row = pgs_clean[pgs_clean['rsID'] == rsid].iloc[0]
        print(f"    {rsid}  chr{row['chr_name']}:{int(row['chr_position'])}  "
              f"effect_allele={row['effect_allele']}  β={row['effect_weight']:.4f}")
else:
    print("    None — all SNPs found positionally.")

print()
print("  SNPs with positional match but NO allele match (multi-allelic/indel):")
if dropped_no_allele_match_rsids:
    for rsid in dropped_no_allele_match_rsids:
        row = pgs_clean[pgs_clean['rsID'] == rsid].iloc[0]
        print(f"    {rsid}  chr{row['chr_name']}:{int(row['chr_position'])}  "
              f"effect_allele={row['effect_allele']}  β={row['effect_weight']:.4f}")
else:
    print("    None — all positionally matched SNPs had a valid allele match.")

print()
print("[PHASE 3 COMPLETE] ✅ Harmonization engine finished.")
print("=" * 65)


  PHASE 3: Multi-File Harmonization Engine
  Strategy: Coordinate-Based Matching + Directional Allele Alignment

  Processing 22 chromosome files...
-----------------------------------------------------------------
  🔬 Chromosome  1 | File: GI_9768_CBR-NIBMG_JointCall_AF_chr1.tsv
     GI variants: 10,351,503  |  PGS SNPs targeting this chr: 17
     Positional matches: 17
     Allele align — Effect=ALT: 9 | Effect=REF(flip): 8 | No match (dropped): 0
  🔬 Chromosome 10 | File: GI_9768_CBR-NIBMG_JointCall_AF_chr10.tsv
     GI variants: 6,371,357  |  PGS SNPs targeting this chr: 9
     Positional matches: 9
     Allele align — Effect=ALT: 5 | Effect=REF(flip): 4 | No match (dropped): 0
  🔬 Chromosome 11 | File: GI_9768_CBR-NIBMG_JointCall_AF_chr11.tsv
     GI variants: 6,453,579  |  PGS SNPs targeting this chr: 11
     Positional matches: 11
     Allele align — Effect=ALT: 7 | Effect=REF(flip): 4 | No match (dropped): 0
  🔬 Chromosome 12 | File: GI_9768_CBR-NIBMG_JointCall_AF_chr12.tsv
   

---
# PHASE 4: Bioinformatic Quality Control

## QC Filter 1 — Palindromic SNP Strand Ambiguity Filter

**What are palindromic SNPs?** SNPs where the two alleles are complementary base pairs:  
**A/T** or **C/G**. On double-stranded DNA, such variants read identically on forward and  
reverse strand — making strand orientation undetectable from sequence alone.

**The risk:** When the PGS and GI datasets were generated using potentially different strand  
conventions, a palindromic SNP might be correctly matched by position but silently have its  
alleles swapped. An effect allele of `A` (freq=0.85 in Indians) could actually correspond  
to `T` in the GI panel if strand reporting differs — inverting the risk frequency to 0.15.

**Solution (frequency-based safeguard):** Palindromic SNPs with `adjusted_risk_freq` between  
**0.42 and 0.58** are dropped. In this zone, frequency alone cannot resolve strand ambiguity  
(both interpretations are plausible). Palindromic SNPs outside this range (e.g., freq=0.1  
or freq=0.9) are **safely retained** because the frequency asymmetry resolves the strand.

**Research basis:** This cutoff (0.42–0.58) is the standard used by the PGS Catalog  
harmonization pipeline, LiftOver, and PLINK's `--flip-scan` function  
(Lambert et al., 2021, *Nature Protocols*).

---

## QC Filter 2 — Minor Allele Frequency (MAF) Filter

**What is MAF?** The frequency of the less-common allele at a given locus.  
Our `adjusted_risk_freq` is oriented to the risk allele, so effective MAF = `min(freq, 1−freq)`.

**Why filter on MAF < 1%?** SNPs with very low risk-allele frequency in Indians:
1. Contribute negligible variance to the PGS in this population.
2. Have highly unstable frequency estimates — a few sequencing errors can dramatically shift the estimate.
3. Are often ancestry-specific rare variants with unreliable GWAS effect weight estimates.

**Research basis:** MAF > 1% is the standard QC threshold used in the Global Biobank  
Meta-analysis Initiative (GBMI) and recommended by the PGS Catalog QC guidelines.


In [ ]:
print("=" * 65)
print("  PHASE 4: Bioinformatic Quality Control")
print("=" * 65)

if len(harmonized_results) == 0:
    raise RuntimeError(
        "[FATAL] No harmonized data to QC! Phase 3 produced zero results. "
        "Check chromosome file formats and paths."
    )

pre_qc_df = pd.concat(harmonized_results, ignore_index=True)
pre_qc_count = len(pre_qc_df)
print(f"  Pre-QC master DataFrame: {pre_qc_count:,} SNPs")

PALINDROMIC_LOW  = 0.42
PALINDROMIC_HIGH = 0.58
MAF_THRESHOLD    = 0.01

PALINDROMIC_PAIRS = [('A','T'), ('T','A'), ('C','G'), ('G','C')]

# ── QC Filter 1: Palindromic SNP Ambiguity ────────────────────────────────
print()
print("-" * 55)
print("[QC FILTER 1] Palindromic SNP Strand Ambiguity Filter")
print("-" * 55)

pre_qc_df['allele_pair'] = list(zip(pre_qc_df['gi_ref_allele'], pre_qc_df['gi_alt_allele']))
mask_palindromic = pre_qc_df['allele_pair'].isin(PALINDROMIC_PAIRS)
mask_ambiguous   = (
    (pre_qc_df['adjusted_risk_freq'] > PALINDROMIC_LOW) &
    (pre_qc_df['adjusted_risk_freq'] < PALINDROMIC_HIGH)
)
mask_palindromic_drop = mask_palindromic & mask_ambiguous

n_palindromic_total   = int(mask_palindromic.sum())
n_palindromic_dropped = int(mask_palindromic_drop.sum())
n_palindromic_kept    = n_palindromic_total - n_palindromic_dropped

print(f"  Total palindromic SNPs detected         : {n_palindromic_total}")
print(f"  Palindromic RETAINED (freq outside zone): {n_palindromic_kept}")
print(f"  Palindromic DROPPED (freq in 0.42-0.58) : {n_palindromic_dropped}")

# Log dropped palindromic SNPs
if n_palindromic_dropped > 0:
    dropped_palind = pre_qc_df[mask_palindromic_drop][['rsID','adjusted_risk_freq']]
    print("  Dropped palindromic SNPs:")
    for _, row in dropped_palind.iterrows():
        print(f"    {row['rsID']}  risk_freq={row['adjusted_risk_freq']:.4f}")
    # Update audit log in pgs_clean
    for rsid in dropped_palind['rsID']:
        pgs_clean.loc[pgs_clean['rsID'] == rsid, 'drop_reason'] = 'palindromic_ambiguous_freq'

post_palindromic_df    = pre_qc_df[~mask_palindromic_drop].copy()
post_palindromic_count = len(post_palindromic_df)
print(f"  SNPs remaining after Palindromic Filter : {post_palindromic_count:,}")

# ── QC Filter 2: MAF Filter ───────────────────────────────────────────────
print()
print("-" * 55)
print("[QC FILTER 2] Minor Allele Frequency (MAF) Filter")
print("-" * 55)

mask_low_maf = post_palindromic_df['adjusted_risk_freq'] < MAF_THRESHOLD
n_low_maf    = int(mask_low_maf.sum())

if n_low_maf > 0:
    dropped_maf = post_palindromic_df[mask_low_maf][['rsID', 'adjusted_risk_freq']]
    print(f"  SNPs dropped (risk_freq < {MAF_THRESHOLD*100:.0f}%): {n_low_maf}")
    for _, row in dropped_maf.iterrows():
        print(f"    {row['rsID']}  risk_freq={row['adjusted_risk_freq']:.6f}")
    for rsid in dropped_maf['rsID']:
        pgs_clean.loc[pgs_clean['rsID'] == rsid, 'drop_reason'] = 'low_maf_below_1pct'
else:
    print(f"  No SNPs dropped by MAF filter (all risk_freq ≥ {MAF_THRESHOLD*100:.0f}%)")

post_maf_df    = post_palindromic_df[~mask_low_maf].copy()
post_maf_count = len(post_maf_df)
print(f"  SNPs remaining after MAF Filter         : {post_maf_count:,}")

# ── QC Summary ──────────────────────────────────────────────────────────
print()
print("-" * 55)
print("[STEP 4.3] QC Filter Impact Summary:")
print(f"  Pre-QC SNP count                       : {pre_qc_count:,}")
print(f"  [−] Dropped by Palindromic Filter       : {n_palindromic_dropped:,}")
print(f"  [−] Dropped by MAF Filter (<1%)         : {n_low_maf:,}")
print(f"  Post-QC SNP count                      : {post_maf_count:,}")
retention = post_maf_count / pre_qc_count * 100 if pre_qc_count > 0 else 0
print(f"  QC Retention Rate                      : {retention:.1f}%")

final_qc_df = post_maf_df
print()
print("[PHASE 4 COMPLETE] ✅ QC filters applied.")
print("=" * 65)


---
# PHASE 5: Final Aggregation, Drop Audit & Data Export

## What the output represents

The **Ancestry-Calibrated Genetic Map** — a curated, QC-filtered table where each row  
represents a CVD risk SNP that has been:

1. **Positionally verified** — confirmed to exist in GenomeIndia at the exact chromosomal coordinate
2. **Directionally aligned** — the risk allele mapped to its correct orientation relative to the Indian reference
3. **Frequency-calibrated** — `adjusted_risk_freq` carries the actual risk allele frequency in Indian ancestry
4. **QC-filtered** — palindromic ambiguities and low-frequency noise removed

## Why we also export the full drop audit log

Reproducibility in genomic research requires documenting **every SNP exclusion and the reason for it**.  
The `dropped_snps_audit_log.csv` provides a complete record of all `INITIAL_PGS_SNP_COUNT` PGS SNPs  
and their fate in this pipeline (Phase 2 printed the actual count for this run — see below; do not  
hardcode a specific number here, it will go stale the moment the input PGS file changes size).  
This is required for the paper's methods section ("SNP harmonization and QC").


In [ ]:
print("=" * 65)
print("  PHASE 5: Final Aggregation & Data Export")
print("=" * 65)

# ── Column map — MUST match NB4 ─────────────────────────────
EXPECTED_COLUMNS = [
    'rsID',
    'chromosome',
    'position_grch',
    'effect_allele',
    'effect_weight_beta',
    'gi_reference_allele',
    'gi_alternate_allele',
    'gi_alt_allele_frequency',
    'indian_ancestry_risk_allele_freq',
]

FINAL_COLUMN_MAP = {
    'rsID': 'rsID',
    'chr_name': 'chromosome',
    'chr_position': 'position_grch',
    'effect_allele': 'effect_allele',
    'effect_weight': 'effect_weight_beta',
    'gi_ref_allele': 'gi_reference_allele',
    'gi_alt_allele': 'gi_alternate_allele',
    'gi_alt_allele_frequency': 'gi_alt_allele_frequency',
    'adjusted_risk_freq': 'indian_ancestry_risk_allele_freq',
}

final_df = final_qc_df[list(FINAL_COLUMN_MAP.keys())].rename(columns=FINAL_COLUMN_MAP).copy()

# Sorting
final_df['chromosome'] = pd.to_numeric(final_df['chromosome'], errors='coerce')
final_df.sort_values(by=['chromosome', 'position_grch'], inplace=True, ignore_index=True)
final_df['chromosome'] = final_df['chromosome'].astype(str)

# Schema validation
assert list(final_df.columns) == EXPECTED_COLUMNS

# ── Save outputs ────────────────────────────────────────────
final_df.to_csv(OUTPUT_PATH, index=False)
print(f"  ✅ Harmonized genetic map saved: {OUTPUT_PATH}")

audit_df = pgs_clean[['rsID', 'chr_name', 'chr_position', 'effect_weight', 'drop_reason']].copy()
audit_df['chr_name'] = audit_df['chr_name'].astype(str)

retained_rsids = set(final_df['rsID'])
audit_df.loc[audit_df['rsID'].isin(retained_rsids), 'drop_reason'] = 'retained_in_final_output'

audit_df.to_csv(DROPPED_SNPS_LOG, index=False)
print(f"  ✅ Drop audit log saved: {DROPPED_SNPS_LOG}")

# ── Final validation ────────────────────────────────────────
assert len(final_df) > 150
assert final_df['effect_weight_beta'].isnull().sum() == 0
assert final_df['indian_ancestry_risk_allele_freq'].isnull().sum() == 0

freq = final_df['indian_ancestry_risk_allele_freq']
assert (freq > 0).all() and (freq < 1).all()

print("\n✅ FINAL VALIDATION PASSED")
print(f"  SNPs retained: {len(final_df)}")

print("\n✅ PIPELINE COMPLETE")
print(f"  📁 {OUTPUT_PATH}")
print(f"  📁 {DROPPED_SNPS_LOG}")